In [0]:
%pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126
%pip install "rc-foundry[all]"

In [0]:
%restart_python

In [0]:
%%sh
# 1. Create the hidden .foundry directory in the driver's home folder
mkdir -p ~/.foundry

# 2. Create the symlink tricking Foundry into using the Volume
ln -s /Volumes/sandbox/model_weights/rfdiffusion3 ~/.foundry/checkpoints

# 3. Verify the link points to the right place
ls -la ~/.foundry

## RFDiffusion3 Pipeline

# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Generation** | RFD3 | Generate novel proteins via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation via Refolding** | RF3 | Predict the structure from designed sequence to validate designability |

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → RF3 (validation) → RMSD comparison
```

---

In [0]:
import os
import warnings
# Set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

In [0]:
import yaml
import os

path_input_dir = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_snake_venom/inputs/"
input_path = os.path.join(path_input_dir, "gopher_alpha_snake_clean_clean.pdb")
# Define a Python dictionary
config_data = {
    'spec_1': {
        'input': input_path,
        'contig': "C1-193,/0,20,D7-13,10,B90-94,20",
        'select_hotspots' : "C9,C10,C11,C12,C13,C14,C46,C47,C48,C70,C72,C73,C74,C75,C76,C77,C78,C79,C112,C113,C114,C116,C143",
        'infer_ori_strategy' : "hotspots"
  }
}

# Define the output YAML file path
output_yaml_filename = 'rfd3_test.yaml'
path_yaml_dir = "/Workspace/Users/karthik.raj@bio-techne.com/MotifScaffold/RFDiffusion3/yaml"
path_yaml = os.path.join(path_yaml_dir, output_yaml_filename)
# Save the dictionary to a YAML file
with open(path_yaml, 'w') as file:
    yaml.dump(config_data, file, default_flow_style=False)

print(f"YAML configuration saved to {path_yaml}")

In [0]:
import os
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Set seed for reproducibility
seed_everything(0)

In [0]:
import shutil
OUTPUT_DIR = "output_designs"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

path_output_dir = os.path.abspath(OUTPUT_DIR)
low_memory_mode = False

!rfd3 design out_dir="{path_output_dir}" inputs="{path_yaml}" prevalidate_inputs=True low_memory_mode={low_memory_mode}

In [0]:
import gzip
import shutil
import biotite
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx

# Extract CIF Files from gzip and view atom_array
paths_structures = []
for file_name in os.listdir(path_output_dir):
  if file_name.endswith(".gz"):
    path_load_gzip = os.path.join(path_output_dir, file_name)
    save_file_name = file_name.replace(".gz", "")
    path_save_structure = os.path.join(path_output_dir, save_file_name)

    with gzip.open(path_load_gzip, 'rb') as f_in:
      with open(path_save_structure, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    paths_structures.append(path_save_structure)
paths_structures

def extract_atom_array(path_structure):
  cif_file = pdbx.CIFFile.read(path_structure)
  atom_array = pdbx.get_structure(cif_file, model=1)
  return atom_array

atom_array = extract_atom_array(paths_structures[0])
view(atom_array)

In [0]:
index = 5
atom_array = extract_atom_array(paths_structures[index])
view(atom_array)

#### Diverging to see whether can generate same inputs to run Sergey's MPNN & AF2 Pipeline

In [0]:
import json
path_example_json = "/Workspace/Users/karthik.raj@bio-techne.com/MotifScaffold/RFDiffusion3/output_designs/rfd3_test_spec_1_0_model_0.json"
with open(path_example_json, 'r') as file:
    data = json.load(file)
data

In [0]:
rfd3_contig = data['specification']['contig']
representation_chain_break = "/0"
contig_target, contig_binder = rfd3_contig.split(f',{representation_chain_break},') 
contig_binder_sergey = []
for contig_region in contig_binder.split(','):
    
    # Diffused Linker regions represented by raw integers (no dashes)
    if "-" not in contig_region:
        contig_binder_sergey.append(f"{contig_region}-{contig_region}")
    else:
        contig_binder_sergey.append(contig_region)
contig_binder_sergey = "/".join(contig_binder_sergey)
contig_binder_sergey

contig_sergey = f"{contig_target}:{contig_binder_sergey}"
contig_sergey

In [0]:
contig_target, contig_binder_sergey

In [0]:
config_data['spec_1']['contig']

In [0]:
import biotite.structure.io.pdb as pdb
dir_name_pdb = "output_designs_pdb"
if os.path.exists(dir_name_pdb):
    shutil.rmtree(dir_name_pdb)
os.makedirs(dir_name_pdb)
for index, path_structure in enumerate(paths_structures):
  atom_array = extract_atom_array(path_structure)
  
  structure_name = f"rfd3_test_model_{index}.pdb"
  path_save_pdb = os.path.join(dir_name_pdb, structure_name)

  pdb_file = pdb.PDBFile()
  pdb_file.set_structure(atom_array)
  pdb_file.write(path_save_pdb)



### Running MPNN via Foundry

In [0]:
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx
print("Designed Structure Chains: ", struc.get_chains(atom_array)) # Target Chain: A, Design Chain: B
seqs, seq_starts = struc.to_sequence(atom_array)


# Create Fixed Residues of the Target
target_len = len(str(seqs[0]))
fixed_residues_target = list(range(1, target_len + 1))
fixed_residues_target = [f"A{res_pos}" for res_pos in fixed_residues_target]

In [0]:
contig_binder = config_data['spec_1']['contig'].split('/0,')[1]
print(contig_binder)

chain_binder = "B"
chain_res_start = 0
fixed_residues_binder = []
for contig_region in contig_binder.split(','):

  # Designed Region of Linker Size
  if not contig_region[0].isalpha():
    chain_res_start += int(contig_region)

  # Fixed Region of Linker Size:
  else:
    if "-" in contig_region:
      start, end = contig_region.split('-')
      delta = int(end) - int(start[1:]) + 1
      fixed_residues_binder.extend(list(range(chain_res_start + 1, chain_res_start + delta + 1)))
      chain_res_start += delta
fixed_residues_binder = [f"{chain_binder}{res_pos}" for res_pos in fixed_residues_binder]
fixed_residues_binder

In [0]:
fixed_residues = fixed_residues_target + fixed_residues_binder

### Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [0]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
OUTPUT_MPNN_DIR = "output_designs_mpnn"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

path_output_mpnn_dir = os.path.abspath(OUTPUT_MPNN_DIR)

engine_config = {
    "model_type": "protein_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": path_output_mpnn_dir,        # Return results in memory
    "write_structures": True,
    "write_fasta": True,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 10,         # Generate 10 sequences per structure
        "remove_waters": True,
        "omit" : ["CYS"],
        "fixed_residues" : fixed_residues
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(input_dicts=input_configs, atom_arrays=[atom_array])

In [0]:
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Extract and display the designed sequences
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

for i, item in enumerate(mpnn_outputs):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts]
    )
    print(f"Sequence {i+1}: {seq_1letter}")


### Section 3: Structure Prediction with RF3 (incompatible with T4)

RF3 (RoseTTAFold 3) predicts protein structures from sequences. By re-folding the MPNN-designed sequence, we can validate whether the design is likely to adopt the intended backbone structure.

**Outputs:** `RF3Output` objects containing:
- `atom_array`: Predicted structure as Biotite AtomArray
- `summary_confidences`: Overall confidence metrics (pLDDT, PAE, pTM, etc.)
- `confidences`: Per-atom/residue confidence scores

**Confidence Metrics:**
| Metric | Description |
|--------|-------------|
| pLDDT | Per-residue confidence (0-1, higher is better) |
| PAE | Predicted Aligned Error (lower is better) |
| pTM | Predicted TM-score |
| ranking_score | Overall model quality score |

In [0]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput

# Initialize RF3 inference engine
inference_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)

# Create input from the MPNN-designed structure (first design)
# This re-folds the sequence to validate it adopts the intended structure
input_structure = InferenceInput.from_atom_array(mpnn_outputs[0].atom_array, example_id="example_protein")
rf3_outputs = inference_engine.run(inputs=input_structure)

# Outputs: dict mapping example_id -> list[RF3Output] (multiple models per input)
print(f"Output keys: {rf3_outputs.keys()}")
print(f"Number of models for 'example_protein': {len(rf3_outputs['example_protein'])}")